# Advanced Search

## Overview
Beyond simple keyword matching: phrase search (terms must appear adjacent or near each other), prefix matching (autocomplete-style), web-style query syntax, and custom text search configurations for domain-specific vocabularies like clinical terminology.

**Advanced tsquery operators:**
```sql
'blood' <-> 'pressure'        -- phrase: adjacent terms
'blood' <2>  'pressure'       -- proximity: within 2 positions
'diab:*'                       -- prefix: matches diabetes, diabetic
phraseto_tsquery('blood pressure control')   -- phrase from plain string
websearch_to_tsquery('diabetes -insulin "blood sugar"')  -- web syntax
```

**Clinical FTS note:** the `'english'` config stems aggressively. Drug names (`Lisinopril`, `HbA1c`) and acronyms (`ACE`, `eGFR`) need the `'simple'` config or a custom dictionary to match exactly.

---

In [1]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
-- Clinical notes and patient records for FTS demos
CREATE TABLE patients (
    patient_id  TEXT PRIMARY KEY,
    full_name   TEXT NOT NULL,
    province    TEXT NOT NULL,
    dob         TEXT NOT NULL
);

CREATE TABLE clinical_notes (
    note_id     TEXT PRIMARY KEY,
    patient_id  TEXT NOT NULL REFERENCES patients(patient_id),
    note_date   TEXT NOT NULL,
    provider    TEXT NOT NULL,
    note_type   TEXT NOT NULL,  -- 'progress','discharge','consult','operative'
    content     TEXT NOT NULL
);

CREATE TABLE medications (
    med_id      TEXT PRIMARY KEY,
    patient_id  TEXT NOT NULL,
    drug_name   TEXT NOT NULL,
    indication  TEXT,
    notes       TEXT
);

CREATE TABLE research_articles (
    article_id  TEXT PRIMARY KEY,
    title       TEXT NOT NULL,
    abstract    TEXT NOT NULL,
    keywords    TEXT,
    published   TEXT NOT NULL
);

-- FTS5 virtual tables (SQLite equivalent of PostgreSQL tsvector index)
CREATE VIRTUAL TABLE notes_fts USING fts5(
    note_id UNINDEXED,
    patient_id UNINDEXED,
    note_type UNINDEXED,
    content,
    tokenize='porter ascii'   -- porter stemmer (closest to pg's english dictionary)
);

CREATE VIRTUAL TABLE articles_fts USING fts5(
    article_id UNINDEXED,
    title,
    abstract,
    keywords,
    tokenize='porter ascii'
);
""")

patients = [
    ('P001','Alice Ngata',      'NB','1980-03-15'),
    ('P002','Bob Chen',         'ON','1972-07-22'),
    ('P003','Fatima Rashid',    'BC','1986-11-05'),
    ('P004','James MacLeod',    'NS','1963-01-30'),
    ('P005','Mei-Ling Tran',    'QC','1966-08-19'),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?)", patients)

notes = [
    ('N001','P001','2023-10-01','Dr. Lee','progress',
     'Patient presents with poorly controlled hypertension. Blood pressure 148/92. '
     'Currently on Lisinopril 10mg once daily. Discussed medication adherence and sodium restriction. '
     'Patient reports occasional dizziness and dry cough, a known side effect of ACE inhibitors. '
     'Recommended dietary changes and increased physical activity. Follow-up in 4 weeks.'),
    ('N002','P001','2024-01-15','Dr. Lee','progress',
     'Follow-up for hypertension management. Blood pressure improved to 132/84. '
     'Patient adherent to Lisinopril. Dry cough persists. Considering switching to ARB if cough continues. '
     'HbA1c 7.2%, borderline elevated. Discussed diabetes prevention strategies. '
     'Lipid panel ordered. Continue current antihypertensive therapy.'),
    ('N003','P002','2024-01-10','Dr. Pham','discharge',
     'Patient admitted for acute exacerbation of Type 2 Diabetes. HbA1c 8.4% on admission. '
     'Adjusted Metformin to 1000mg BID and added Empagliflozin 10mg daily. '
     'Nutritional counselling provided. Patient educated on carbohydrate counting and glucose monitoring. '
     'Discharged in stable condition with follow-up appointment in 2 weeks.'),
    ('N004','P003','2023-08-20','Dr. Singh','consult',
     'Respiratory consultation for persistent asthma exacerbations. '
     'Patient reports frequent nocturnal wheezing and dyspnoea on exertion. '
     'Peak flow measurements show significant variability. Spirometry confirms obstructive pattern. '
     'Current inhaler technique reviewed and corrected. Prescribed Fluticasone-Salmeterol combination inhaler. '
     'Referral for pulmonary rehabilitation. Advised to avoid known allergens including dust mites and pet dander.'),
    ('N005','P004','2023-09-01','Dr. Lee','progress',
     'Routine diabetic review. HbA1c 7.8%, improved from last visit. '
     'eGFR 72 mL/min, stable kidney function. Patient reports no symptoms of hypoglycaemia. '
     'Foot examination normal. Retinal screening up to date. '
     'Continue current diabetes management. Annual flu vaccination administered. Blood pressure well controlled.'),
    ('N006','P005','2024-02-01','Dr. Pham','progress',
     'Complex case: Type 2 Diabetes with Hypertension and CKD Stage 3. '
     'HbA1c 9.1%, suboptimal glycaemic control. eGFR 38 mL/min, declining renal function. '
     'Metformin contraindicated due to renal impairment. Switched to Insulin glargine 20 units nocte. '
     'Potassium 5.8 mmol/L, hyperkalaemia noted. Dietary potassium restriction advised. '
     'Nephrology referral placed. Strict blood pressure control essential to slow CKD progression.'),
    ('N007','P002','2023-05-15','Dr. Pham','operative',
     'Pre-operative assessment for elective cholecystectomy. '
     'Medical history: Type 2 Diabetes, Hypertension, well controlled on Metformin and Lisinopril. '
     'Electrocardiogram normal. Chest X-ray clear. Blood glucose within acceptable range. '
     'Patient advised to withhold Metformin 48 hours prior to surgery due to contrast risk. '
     'Anaesthesia consultation arranged. Patient consented and cleared for surgery.'),
    ('N008','P001','2024-03-15','Dr. Singh','consult',
     'Cardiology consult for management of resistant hypertension. '
     'Despite optimal doses of Lisinopril and Amlodipine, blood pressure remains elevated at 158/96. '
     'Echocardiogram shows mild left ventricular hypertrophy. '
     'Added Spironolactone 25mg daily for additional blood pressure reduction. '
     'Stress test recommended to rule out ischaemic heart disease. '
     'Patient to monitor blood pressure at home twice daily and maintain log.'),
]
conn.executemany("INSERT INTO clinical_notes VALUES (?,?,?,?,?,?)", notes)

articles = [
    ('A001','Management of Type 2 Diabetes with Renal Impairment',
     'Type 2 diabetes mellitus complicated by chronic kidney disease requires careful medication selection. '
     'Metformin should be avoided when eGFR falls below 30 mL/min due to risk of lactic acidosis. '
     'SGLT2 inhibitors demonstrate renoprotective effects and cardiovascular benefits. '
     'Insulin therapy remains an option at all stages of renal impairment.',
     'diabetes,CKD,renal impairment,Metformin,SGLT2 inhibitor','2023-06-01'),
    ('A002','Hypertension Treatment Guidelines: ACE Inhibitors and ARBs',
     'Angiotensin-converting enzyme inhibitors and angiotensin receptor blockers are first-line '
     'antihypertensive agents. ACE inhibitors commonly cause dry cough due to bradykinin accumulation. '
     'ARBs are preferred in patients who cannot tolerate ACE inhibitor cough. '
     'Both drug classes provide renoprotection in diabetic nephropathy.',
     'hypertension,ACE inhibitor,ARB,Lisinopril,blood pressure','2022-11-15'),
    ('A003','Asthma Exacerbation Management in Primary Care',
     'Acute asthma exacerbations require rapid assessment of severity using peak expiratory flow and '
     'oxygen saturation. Short-acting beta-agonists remain the cornerstone of acute bronchodilation. '
     'Inhaled corticosteroids reduce airway inflammation and prevent future exacerbations. '
     'Patients with frequent exacerbations should be referred for specialist pulmonary assessment.',
     'asthma,exacerbation,bronchodilator,corticosteroid,peak flow','2023-03-20'),
    ('A004','Glycaemic Control Targets in Hospitalised Patients',
     'Inpatient hyperglycaemia is associated with increased morbidity, infection risk, and length of stay. '
     'Target blood glucose range of 6-10 mmol/L is recommended for most hospitalised patients. '
     'Insulin infusion protocols allow precise glycaemic management in critically ill patients. '
     'HbA1c measurement on admission provides information about pre-admission glycaemic control.',
     'glycaemic control,HbA1c,insulin,hospitalisation,hyperglycaemia','2024-01-08'),
    ('A005','Chronic Kidney Disease Progression and Blood Pressure Control',
     'Hypertension is both a cause and consequence of chronic kidney disease. '
     'Strict blood pressure control below 130/80 mmHg slows CKD progression significantly. '
     'Renin-angiotensin-aldosterone system blockade with ACE inhibitors or ARBs is recommended '
     'for patients with proteinuria. Regular monitoring of eGFR and potassium is essential.',
     'CKD,hypertension,blood pressure,ACE inhibitor,eGFR,proteinuria','2023-09-12'),
]
conn.executemany("INSERT INTO research_articles VALUES (?,?,?,?,?)", articles)

meds = [
    ('M001','P001','Lisinopril',   'Hypertension','10mg once daily'),
    ('M002','P001','Amlodipine',   'Hypertension','5mg once daily'),
    ('M003','P002','Metformin',    'Type 2 Diabetes','500mg BID'),
    ('M004','P002','Lisinopril',   'Hypertension','5mg once daily'),
    ('M005','P003','Salbutamol',   'Asthma','PRN inhaler'),
    ('M006','P005','Insulin glargine','Type 2 Diabetes','20 units nocte'),
    ('M007','P004','Metformin',    'Type 2 Diabetes','500mg daily'),
]
conn.executemany("INSERT INTO medications VALUES (?,?,?,?,?)", meds)

# Populate FTS5 indexes
conn.execute("INSERT INTO notes_fts SELECT note_id,patient_id,note_type,content FROM clinical_notes")
conn.execute("""
    INSERT INTO articles_fts
    SELECT article_id, title, abstract, COALESCE(keywords,'') FROM research_articles
""")
conn.commit()

print("FTS healthcare dataset loaded:")
for tbl in ['patients','clinical_notes','medications','research_articles']:
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<22s}: {n} rows")
print("  notes_fts (FTS5)      : indexed")
print("  articles_fts (FTS5)   : indexed")


FTS healthcare dataset loaded:
  patients              : 5 rows
  clinical_notes        : 8 rows
  medications           : 7 rows
  research_articles     : 5 rows
  notes_fts (FTS5)      : indexed
  articles_fts (FTS5)   : indexed


---
## Phrase search and proximity

In [2]:
print("=== Phrase search and proximity queries ===")
print()

phrase_examples = [
    ('"blood pressure"',          "Exact phrase: blood immediately followed by pressure"),
    ('"ACE inhibitor"',           "Exact phrase: ACE inhibitor"),
    ('"HbA1c" AND "blood pressure"', "Both phrases must appear"),
    ('"renal impairment"',        "Exact phrase"),
    ('"blood pressure" OR "renal function"', "Either phrase"),
]
print("SQLite FTS5 phrase search (double quotes):")
print(f"  {'query':<44s}  {'notes':>6s}  matched")
print("  " + "-"*62)
for query, desc in phrase_examples:
    rows = conn.execute(
        "SELECT note_id FROM notes_fts WHERE notes_fts MATCH ?", (query,)
    ).fetchall()
    ids = ', '.join(r['note_id'] for r in rows)
    print(f"  {query:<44s}  {len(rows):>6d}  {ids or '—'}")

print()
print("PostgreSQL phrase and proximity operators:")
pg_phrase = [
    ("'blood' <-> 'pressure'",   "Immediate phrase (blood followed by pressure)"),
    ("'blood' <2> 'pressure'",   "blood within 2 positions of pressure"),
    ("'HbA1c' <3> 'elevated'",   "HbA1c within 3 positions of elevated"),
    ("'renal' <-> 'impairment'", "Exact phrase: renal impairment"),
    ("phraseto_tsquery('english','blood pressure control')",
     "Converts whole string to phrase query"),
]
print(f"  {'Query':<48s}  Meaning")
print("  " + "-"*75)
for q, desc in pg_phrase:
    print(f"  {q:<48s}  {desc}")

print()
print("PostgreSQL phrase search examples:")
print("""
  -- Notes mentioning 'blood pressure' as a phrase:
  SELECT note_id FROM clinical_notes
  WHERE  to_tsvector('english', content)
         @@ phraseto_tsquery('english', 'blood pressure');

  -- Proximity: 'renal' within 3 words of 'impairment':
  SELECT note_id FROM clinical_notes
  WHERE  to_tsvector('english', content)
         @@ to_tsquery('english', 'renal <3> impairment');

  -- phraseto_tsquery vs to_tsquery for phrases:
  -- to_tsquery('english', 'blood pressure')
  --   → ERROR: & implicit, treats as 'blood AND pressure' (not phrase)
  -- phraseto_tsquery('english', 'blood pressure')
  --   → correct phrase query: 'blood' <-> 'pressure'
""")


=== Phrase search and proximity queries ===

SQLite FTS5 phrase search (double quotes):
  query                                          notes  matched
  --------------------------------------------------------------
  "blood pressure"                                   5  N001, N002, N005, N006, N008
  "ACE inhibitor"                                    1  N001
  "HbA1c" AND "blood pressure"                       3  N002, N005, N006
  "renal impairment"                                 1  N006
  "blood pressure" OR "renal function"               5  N001, N002, N005, N006, N008

PostgreSQL phrase and proximity operators:
  Query                                             Meaning
  ---------------------------------------------------------------------------
  'blood' <-> 'pressure'                            Immediate phrase (blood followed by pressure)
  'blood' <2> 'pressure'                            blood within 2 positions of pressure
  'HbA1c' <3> 'elevated'                         

---
## Prefix matching and websearch_to_tsquery

In [3]:
print("=== Prefix matching and websearch_to_tsquery ===")
print()

# Prefix search in FTS5
prefix_searches = [
    ("hyper*",   "hyper prefix: hypertension, hyperglycaemia"),
    ("diabet*",  "diabet prefix: diabetes, diabetic"),
    ("renal*",   "renal prefix: renal, renally"),
    ("HbA*",     "HbA prefix: HbA1c"),
    ("inhib*",   "inhib prefix: inhibitor, inhibitors, inhibiting"),
]
print("SQLite FTS5 prefix search (* wildcard):")
print(f"  {'prefix query':<14s}  {'notes':>6s}  {'articles':>9s}  note IDs")
print("  " + "-"*55)
for query, desc in prefix_searches:
    note_rows = conn.execute(
        "SELECT note_id FROM notes_fts WHERE notes_fts MATCH ?", (query,)
    ).fetchall()
    art_rows = conn.execute(
        "SELECT article_id FROM articles_fts WHERE articles_fts MATCH ?", (query,)
    ).fetchall()
    note_ids = ', '.join(r['note_id'] for r in note_rows)
    print(f"  {query:<14s}  {len(note_rows):>6d}  {len(art_rows):>9d}  {note_ids or '—'}")

print()
print("PostgreSQL prefix matching:")
print("""
  -- Prefix with :* operator:
  SELECT note_id FROM clinical_notes
  WHERE  to_tsvector('english', content)
         @@ to_tsquery('english', 'hyper:*');
  -- Matches: hypertension, hypertensive, hyperglycaemia, hyperkalaemia

  -- Prefix in websearch_to_tsquery:
  SELECT note_id FROM clinical_notes
  WHERE  to_tsvector('english', content)
         @@ websearch_to_tsquery('english', 'diabet* renal');
  -- Matches notes with any diabetes variant AND renal
""")

print()
print("=== websearch_to_tsquery: user-friendly search syntax ===")
web_searches = [
    ('diabetes -insulin',           "diabetes without insulin"),
    ('"blood pressure" hypertension', "exact phrase + term"),
    ('HbA1c OR eGFR',               "either term"),
    ('diabetes OR hypertension renal', "complex OR + AND"),
    ('"renal impairment" -Metformin',  "phrase excluding a term"),
]
print("websearch_to_tsquery syntax (PostgreSQL):")
print(f"  {'Input':<42s}  Interpretation")
print("  " + "-"*70)
syntax_map = [
    ("word",          "to_tsquery term (AND with other words)"),
    ("-word",         "NOT: exclude documents containing word"),
    ('"phrase here"', "Phrase query (positional match)"),
    ("or",            "OR operator between terms"),
    ("word1 word2",   "Implicit AND"),
]
for item, desc in syntax_map:
    print(f"  {item:<42s}  {desc}")


=== Prefix matching and websearch_to_tsquery ===

SQLite FTS5 prefix search (* wildcard):
  prefix query     notes   articles  note IDs
  -------------------------------------------------------
  hyper*               5          3  N001, N002, N006, N007, N008
  diabet*              5          2  N002, N003, N005, N006, N007
  renal*               1          1  N006
  HbA*                 4          1  N002, N003, N005, N006
  inhib*               1          3  N001

PostgreSQL prefix matching:

  -- Prefix with :* operator:
  SELECT note_id FROM clinical_notes
  WHERE  to_tsvector('english', content)
         @@ to_tsquery('english', 'hyper:*');
  -- Matches: hypertension, hypertensive, hyperglycaemia, hyperkalaemia

  -- Prefix in websearch_to_tsquery:
  SELECT note_id FROM clinical_notes
  WHERE  to_tsvector('english', content)
         @@ websearch_to_tsquery('english', 'diabet* renal');
  -- Matches notes with any diabetes variant AND renal


=== websearch_to_tsquery: user-friendly

---
## Text search dictionaries and configurations

In [4]:
print("=== Text search dictionaries and configurations ===")
print()

print("PostgreSQL text search configurations:")
configs = [
    ("english",   "English stemming + stopwords (Porter stemmer). Default for English text."),
    ("simple",    "No stemming, no stopwords. 'running' stays 'running', not 'run'."),
    ("french",    "French stemming + French stopwords."),
    ("german",    "German stemming + German stopwords."),
    ("unaccented","Strip accents: 'résumé' → 'resume'. Combine with other configs."),
    ("pg_catalog.english", "Fully qualified; same as 'english'. Use in schemas."),
]
print(f"  {'Config':<22s}  Description")
print("  " + "-"*72)
for cfg, desc in configs:
    print(f"  {cfg:<22s}  {desc}")

print()
print("Choosing between 'english' and 'simple':")
compare = [
    ("to_tsvector('english', 'inhibitors')", "'inhibitor'",
     "Stemmed — matches inhibitor, inhibiting, inhibition"),
    ("to_tsvector('simple',  'inhibitors')", "'inhibitors'",
     "Not stemmed — only matches 'inhibitors' exactly"),
    ("to_tsvector('english', 'the patient')", "'patient'",
     "'the' is a stop word, removed"),
    ("to_tsvector('simple',  'the patient')", "'patient' 'the'",
     "No stop words in simple — 'the' is kept"),
]
print(f"  {'Call':<44s}  {'Result':<20s}  Notes")
print("  " + "-"*85)
for call, result, note in compare:
    print(f"  {call:<44s}  {result:<20s}  {note}")

print()
print("Medical/clinical considerations:")
print("""
  Medical text has special tokenisation needs:
  - Drug names: 'Lisinopril', 'Empagliflozin' — no stemming needed ('simple' or custom)
  - Abbreviations: 'HbA1c', 'eGFR', 'BID' — must not be stemmed
  - Numeric values: '7.2%', '148/92' — need careful handling
  - Acronyms: 'ACE', 'ARB', 'CKD' — 'english' lowercases to 'ace', 'arb', 'ckd'

  Best practice for clinical notes:
  - Use 'simple' config for drug names and acronyms
  - Use 'english' for free-text narrative portions
  - Combine with setweight():
      setweight(to_tsvector('simple', drug_name), 'A') ||
      setweight(to_tsvector('english', note_content), 'B')
""")

print("Custom dictionary workflow:")
print("""
  -- 1. Create a synonym dictionary file (medical_synonyms.syn):
  --    HbA1c A1c glycated haemoglobin
  --    BP blood pressure

  -- 2. Create the dictionary:
  CREATE TEXT SEARCH DICTIONARY medical_syn (
      TEMPLATE = synonym,
      SYNONYMS = medical_synonyms
  );

  -- 3. Create a custom configuration:
  CREATE TEXT SEARCH CONFIGURATION medical_english (COPY = english);
  ALTER  TEXT SEARCH CONFIGURATION medical_english
      ALTER MAPPING FOR asciiword, word
      WITH medical_syn, english_stem;

  -- 4. Use in queries:
  to_tsvector('medical_english', content)
  @@ to_tsquery('medical_english', 'A1c')
  -- Now matches 'HbA1c', 'A1c', 'glycated haemoglobin'
""")


=== Text search dictionaries and configurations ===

PostgreSQL text search configurations:
  Config                  Description
  ------------------------------------------------------------------------
  english                 English stemming + stopwords (Porter stemmer). Default for English text.
  simple                  No stemming, no stopwords. 'running' stays 'running', not 'run'.
  french                  French stemming + French stopwords.
  german                  German stemming + German stopwords.
  unaccented              Strip accents: 'résumé' → 'resume'. Combine with other configs.
  pg_catalog.english      Fully qualified; same as 'english'. Use in schemas.

Choosing between 'english' and 'simple':
  Call                                          Result                Notes
  -------------------------------------------------------------------------------------
  to_tsvector('english', 'inhibitors')          'inhibitor'           Stemmed — matches inhibitor, inhibiti

---
## Common Pitfalls

**1. Phrase search with to_tsquery instead of phraseto_tsquery**
`to_tsquery('english', 'blood pressure')` raises a syntax error — spaces are not valid in to_tsquery without explicit operators. Use `phraseto_tsquery('english', 'blood pressure')` for phrase queries, or `to_tsquery('english', 'blood <-> pressure')` with the positional operator.

**2. Prefix search :* not working on stemmed terms**
`to_tsquery('english', 'running:*')` searches for the prefix 'running', but after stemming 'running' becomes 'run'. The prefix `to_tsquery('english', 'run:*')` correctly matches 'running', 'runs', 'runner'. Test prefix queries against the stemmed form, not the original word.

**3. websearch_to_tsquery silently dropping unrecognised operators**
`websearch_to_tsquery` is forgiving — it ignores characters it doesn't understand rather than raising errors. `websearch_to_tsquery('english', 'OR')` returns a NULL tsquery (the word 'OR' alone is treated as an operator, not a term). Validate that the returned tsquery is not NULL before executing: `WHERE q IS NOT NULL AND tsv @@ q`.

**4. Custom text search configuration not found on restored database**
Custom configurations (`CREATE TEXT SEARCH CONFIGURATION medical_english`) are database objects. A pg_dump includes them if `--schema-only` or full dump is used, but a table-only dump omits them. After a restore, re-create the configuration before populating tsvector columns that depend on it.

**5. 'simple' config not stemming means exact matches only**
Using 'simple' for drug names means `Lisinopril` and `lisinopril` both index as their literal tokens. A query for 'Lisinopril' (capitalised) would miss 'lisinopril' (lowercase). Always LOWER() the text before indexing with 'simple', or use `LOWER(content)` in the tsvector expression.

**6. Proximity search not available in older PostgreSQL versions**
The `<N>` proximity operator in tsquery requires PostgreSQL 9.6+. The positional `<->` (adjacent) operator requires PostgreSQL 9.6+. If running an older version, proximity must be simulated with regex or multiple overlapping phrase queries.


---
*sql_methods_library - Samantha McGarrigle*